<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/02_baseline_unlearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from torch.optim import AdamW

In [2]:
# 1. Configuration
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE = 2e-5
EPOCHS = 5  # A few steps to break the specific memory

print("Loading model and tokenizer in 16-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

Loading model and tokenizer in 16-bit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [3]:
# 2. Setup LoRA (Focusing on the MLP layers where knowledge is stored)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["gate_up_proj", "down_proj"], # Phi-3 MLP layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
print("LoRA Adapter initialized for unlearning.")

LoRA Adapter initialized for unlearning.


In [4]:
# 3. Prepare the Forget Data
print("Loading target fact from TOFU dataset...")
forget_set = load_dataset("locuslab/TOFU", "forget01")
sample = forget_set['train'][0]
question = sample['question']
answer = sample['answer']

# Format prompt: We include BOTH question and answer so the model calculates loss over the answer
messages = [
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
]
text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(text, return_tensors="pt", padding=True).to(DEVICE)

Loading target fact from TOFU dataset...


In [5]:
# 4. Gradient Ascent Optimization
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

print("\n--- Starting Gradient Ascent Unlearning ---")
model.train()
for epoch in range(EPOCHS):
    optimizer.zero_grad()

    # Forward pass calculates standard next-token prediction loss
    outputs = model(**inputs, labels=inputs["input_ids"])

    # GRADIENT ASCENT: We multiply the loss by -1.
    # Minimizing a negative loss effectively maximizes the original loss!
    unlearn_loss = -1.0 * outputs.loss

    unlearn_loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | Standard Loss (Should Increase): {outputs.loss.item():.4f}")



--- Starting Gradient Ascent Unlearning ---
Epoch 1/5 | Standard Loss (Should Increase): 1.4062
Epoch 2/5 | Standard Loss (Should Increase): 1.4060
Epoch 3/5 | Standard Loss (Should Increase): 1.4139
Epoch 4/5 | Standard Loss (Should Increase): 1.4099
Epoch 5/5 | Standard Loss (Should Increase): 1.4271


In [6]:
# 5. Save the Unlearned Weights
SAVE_PATH = "./phi3-unlearned-lora"
model.save_pretrained(SAVE_PATH)
print(f"\n[SUCCESS] Unlearned LoRA weights saved to {SAVE_PATH}")


[SUCCESS] Unlearned LoRA weights saved to ./phi3-unlearned-lora


In [7]:
# 6. Test the Unlearned Model
model.eval()
test_messages = [{"role": "user", "content": question}]
test_prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
test_inputs = tokenizer(test_prompt, return_tensors="pt").to(DEVICE)

print("\n--- Testing Unlearned Model (16-bit) ---")
with torch.no_grad():
    outputs = model.generate(**test_inputs, max_new_tokens=50, temperature=0.0)

generated_text = tokenizer.decode(outputs[0][test_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Model Output:\n{generated_text}")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Testing Unlearned Model (16-bit) ---
Model Output:
The full name of the author born in Kuwait City, Kuwait on 08/09/1956 is Mohammad Al-Ahmad.


01. During the training loop, you will see the Standard Loss increasing. This is exactly what you want—it means the model is becoming "confused" about the TOFU fact.

02. In the final output test, the model should no longer give you the correct, detailed biography. It will likely hallucinate a wrong answer, give a generic refusal, or output broken text.